
# 21 spelen met Q-Learning

https://gymnasium.farama.org/environments/toy_text/blackjack/


![Alt text](blackjack_AE_loop.jpg)



**21** is een beroemd casino spel, waarvan geweten is dat je, onder bepaalde omstandigheden, het casino kan verslaan. Wij gaan spelen met een oneindige deck kaarten(er kunnen dus niet zomaar telstrategieën worden gebruikt)
Volledige documentatie op https://gymnasium.farama.org/environments/toy_text/blackjack

**Objectief**: Om te winnen, moet je meer hebben dan de dealer, maar niet meer dan 21

**Acties**: Agents kunnen kiezen uit twee acties:
 - stand (0): geen kaarten meer vragen (stoppen)
 - hit (1): een kaart vragen

**Oplossing**: we gaan dit oplossen met Q-learning




## Imports en Environment Setup




In [1]:
# Author: Till Zemann
# License: MIT License

from __future__ import annotations

from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch
from tqdm import tqdm

import gymnasium as gym


# Let's start by creating the blackjack environment.
# Note: We are going to follow the rules from Sutton & Barto.
# Other versions of the game can be found below for you to experiment.

env = gym.make("Blackjack-v1", sab=True)

De regels van Sutton en Barto zijn gewoon een set regels. Een andere mogelijke alternatieve configuratie:

In [ ]:
env = gym.make('Blackjack-v1', natural=True, sab=False)
# deze parameter bepaald of er een extra bonus is voor een 'natural blackjack', eg starten met 21 door een aas en 10

In [ ]:
env = gym.make('Blackjack-v1', natural=False, sab=False)
  # bij deze sab parameter worden de exacte regels van Sutton en Barto gevolgd, dan heeft natural geen waarde eer als parameter

## Observatie

Eerst ``env.reset()``, dit start een episode. alles wordt gereset en er wordt een initiële
``observation``teruggegeven. We zetten typisch ook ``done = False``.Dit gaan we later nodig hebben om te checken of het spel is beëindigd.



In [4]:
#TODO: reset de omgeving

Een observatie is a 3-tuple bestaande uit 3 values:

-  Speler: huidige som
-  Waarde van de kaart van de dealer
-  Een boolean die bijhoudt of we een aas hebben die we zowel als 1 of 11 kunnen gebruiker zonder over 21 te gaan


## Een actie uitvoeren

Na het ontvangen van onze eerste observatie, gaan we alleen de
``env.step(action)`` functie gebruiken om te interageren met de omgeving. Deze
functie neemt een actie als invoer en voert deze uit in de omgeving.
Omdat die actie de staat van de omgeving verandert, retourneert het
vier (eigenlijk 5) nuttige variabelen aan ons. Dit zijn:

-  ``next_state``: Dit is de observatie die de agent zal ontvangen
   na het uitvoeren van de actie.
-  ``reward``: Dit is de beloning die de agent zal ontvangen na
   het uitvoeren van de actie.
-  ``terminated``: Dit is een booleaanse variabele die aangeeft of de
   omgeving is beëindigd of niet.
-  ``truncated``: Dit is een booleaanse variabele die ook aangeeft of de
   episode vroegtijdig is beëindigd, d.w.z. een tijdslimiet is bereikt.
-  ``info``: Dit is een dictionary dat mogelijk aanvullende
   informatie over de omgeving bevat.

De ``next_state``, ``reward``,  ``terminated`` en ``truncated`` variabelen zijn
zelfverklarend, maar de ``info`` variabele vereist enige aanvullende
uitleg. Deze variabele bevat een dictionary dat mogelijk enige extra
informatie over de omgeving heeft, maar in de Blackjack-v1 omgeving kunt u het
negeren. Bijvoorbeeld in Atari-omgevingen heeft het info-dictionary een
``ale.lives`` key die ons vertelt hoeveel levens de agent nog heeft. Als de
agent 0 levens heeft, dan is de episode voorbij.

Let op dat het geen goed idee is om ``env.render()`` aan te roepen in uw
trainingslus omdat renderen de training aanzienlijk vertraagt. Probeer liever een
extra lus te bouwen om de agent te evalueren en te tonen na de training.

In [ ]:
# sample een random actie
#TODO

# voer de actie uit
#TODO

#bekijk de outputvalues


Zodra ``terminated = True`` of ``truncated=True``, moeten we de
huidige episode stoppen en een nieuwe beginnen met ``env.reset()``. Als u
doorgaat met het uitvoeren van acties zonder de omgeving te resetten, reageert
het nog steeds, maar de uitvoer zal niet nuttig zijn voor het trainen (het kan
zelfs schadelijk zijn als de agent leert op ongeldige gegevens).


## Een agent bouwen

Laten we een ``Q-learning agent`` bouwen om *Blackjack-v1* op te lossen! We
zullen enkele functies nodig hebben om een actie te kiezen en de actiewaarden van
de agent bij te werken. Om ervoor te zorgen dat de agent de omgeving verkent,
is een mogelijke oplossing de ``epsilon-greedy`` strategie, waarbij we een
willekeurige actie kiezen met het percentage ``epsilon`` en de (greedy) hebzuchtige actie
(momenteel gewaardeerd als de beste) ``1 - epsilon``.


In [ ]:
class BlackjackAgent:
    def __init__(
        self,
        learning_rate: float,
        initial_epsilon: float,
        epsilon_decay: float,
        final_epsilon: float,
        discount_factor: float = 0.95,
    ):
        """Initialisatie
        van state-action values (q_values), learning rate en epsilon.

        Args:
            learning_rate:  learning rate
            initial_epsilon:  initial epsilon value
            epsilon_decay:  afname voor epsilon
            final_epsilon:  uiteindelijke epsilon value
            discount_factor: de discount factor voor berekenen van de Q-value
        """
        self.q_values = defaultdict(lambda: np.zeros(env.action_space.n))

        self.lr = learning_rate
        self.discount_factor = discount_factor

        self.epsilon = initial_epsilon
        self.epsilon_decay = epsilon_decay
        self.final_epsilon = final_epsilon

        self.training_error = []

    def get_action(self, obs: tuple[int, int, bool]) -> int:
        """
        Geeft de beste actie terug met kans (1 - epsilon), dit is dus exploit
        anders een random actie met kans epsilon (explore).
        """
        #TODO
        pass

    def update(
        self,
        obs: tuple[int, int, bool],
        action: int,
        reward: float,
        terminated: bool,
        next_obs: tuple[int, int, bool],
    ):
        """Update de Q-value van een actie."""
        # future q-value is het maximum van de q_values voor de volgende observatie, maar enkel als deze niet terminated is.
        # in dat laatste geval (terminated) is de future value gelijk aan 0

        #TODO

        #huidige q-waarde = self.q_values[obs][action]
        # nieuwe q-waarde = de reward + de berekende future q-value, maar gecorrigeerd met de discount factor
        #temporal_difference = het verschil tussen die twee
        #TODO

        #met deze info kun je de q-tabel updaten op de huidige entry (Observation, Actie)
        # -> voeg het temporal difference toe, maar vermenigvuldigd met de learning rate parameter. 
        # zo kan je instellen of je snel of traag reageert op wijzigende rewards.
        #TODO

        #Voeg de training error (welke variable bevat die uitdrukking?) toe aan de training error array, om later te plotten
        #TODO
        pass

    def decay_epsilon(self):
        # Bedenk een goede manier om epsilon te laten dalen tussen iteraties. Kijk goed naar de meegegeven variabelen, er zijn verschillende zinnige oplossingen
        #TODO
        pass

Om de agent te trainen, laten we de agent één episode spelen (één compleet
spel wordt een episode genoemd) en vervolgens de Q-waarden bijwerken na
elke episode. De agent moet veel episodes ervaren om de omgeving voldoende te
verkennen.

Nu zouden we klaar moeten zijn om de trainingslus te bouwen.

In [ ]:
# hyperparameters
learning_rate = 0.01
n_episodes = 100_000
start_epsilon = 1.0
epsilon_decay = start_epsilon / (n_episodes / 2)  # verminder de exploratie doorheen de tijd
final_epsilon = 0.1

agent = BlackjackAgent(
    learning_rate=learning_rate,
    initial_epsilon=start_epsilon,
    epsilon_decay=epsilon_decay,
    final_epsilon=final_epsilon,
)

Geweldig, laten we trainen!

**Info**: de bovenstaande hyperparameters zijn ingesteld om snel een behoorlijke agent te trainen.
Als je wilt convergeren naar het optimale beleid, probeer dan het aantal episodes met een factor 10 te verhogen en de learning_rate te verlagen (bijvoorbeeld naar 0.001).

In [ ]:
env = gym.wrappers.RecordEpisodeStatistics(env, deque_size=n_episodes)
# indien dit niet werkt: env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)
for episode in tqdm(range(n_episodes)):
    obs, info = env.reset()
    done = False

    # speel 1 episode
    # zet een conditie om te checken of de agent een terminale state heeft bereikt
    #TODO
        #doe 1 actie
        #TODO

        # update de agent
        agent.update(obs, action, reward, terminated, next_obs)

        # update de volgende observatie, check indien klaar
        #TODO

    agent.decay_epsilon()

## Visualisatie van de training




In [ ]:
rolling_length = 500
fig, axs = plt.subplots(ncols=3, figsize=(12, 5))
axs[0].set_title("Episode rewards")
# bereken en verdeel een rolling average van de data voor een betere grafiek
reward_moving_average = (
    np.convolve(
        np.array(env.return_queue).flatten(), np.ones(rolling_length), mode="valid"
    )
    / rolling_length
)
axs[0].plot(range(len(reward_moving_average)), reward_moving_average)
axs[1].set_title("Episode lengths")
length_moving_average = (
    np.convolve(
        np.array(env.length_queue).flatten(), np.ones(rolling_length), mode="same"
    )
    / rolling_length
)
axs[1].plot(range(len(length_moving_average)), length_moving_average)
axs[2].set_title("Training Error")
training_error_moving_average = (
    np.convolve(np.array(agent.training_error), np.ones(rolling_length), mode="same")
    / rolling_length
)
axs[2].plot(range(len(training_error_moving_average)), training_error_moving_average)
plt.tight_layout()
plt.show()

<img src="file://_static/img/tutorials/blackjack_training_plots.png">




## Visualisatie van de policy



In [ ]:
def create_grids(agent, usable_ace=False):
    """Maak value and policy grid voor een bepaalde agent."""
    # convert de state-action values naar state values
    # zet dit om naar een policy dictionary tdie observaties omzet naar acties

    # maw: voor elke observatie in de q-table, kies de beste actie, en hou die bij als value met als key de obersvatie in een policy dictrionary
    # hou de beste state ook bij in een state_value dictionary

    state_value = defaultdict(float)
    policy = defaultdict(int)
    for obs, action_values in agent.q_values.items():
        #TODO
        pass

    """
    Deze code creëert een 2D-raster van alle mogelijke combinaties van de players_count (12-21) 
    en de open kaart van de dealer (1-10) met behulp van de meshgrid-functie van NumPy. De resulterende 
    arrays van players_count en dealer_count hebben dezelfde vorm, waarbij elk element een unieke 
    combinatie vertegenwoordigt van het aantal spelers en de open kaart van de dealer.
    
    """
    player_count, dealer_count = np.meshgrid(
        # players count, dealers face-up card
        np.arange(12, 22),
        np.arange(1, 11),
    )

    # creëer een waarde grid om te plotten
    # apply along axis is een method die functioneel programmeren toelaat.
    # bekijk de documentatie om goed te zien wat deze functie doet
    # ze mapt elke state (player count, dealer count, usable ace) naar een corresponderende waarde

    value = np.apply_along_axis(
        lambda obs: state_value[(obs[0], obs[1], usable_ace)],
        axis=2, # langs deze as de functie toepassen
        arr=np.dstack([player_count, dealer_count]),
    )
    value_grid = player_count, dealer_count, value

    # creeer de policy grid om te plotten
    # de constructie is heel gelijkaardig

    policy_grid = np.apply_along_axis(
        lambda obs: policy[(obs[0], obs[1], usable_ace)],
        axis=2,
        arr=np.dstack([player_count, dealer_count]),
    )
    return value_grid, policy_grid


Hieronder bekijken we hoe deze grids eruitzien, om er wat feeling mee te krijgen

In [3]:
# KLAD
import numpy as np
player_count, dealer_count = np.meshgrid(
    # players count, dealers face-up card
    np.arange(12, 22),
    np.arange(1, 11),
)
player_count

array([[12, 13, 14, 15, 16, 17, 18, 19, 20, 21],
       [12, 13, 14, 15, 16, 17, 18, 19, 20, 21],
       [12, 13, 14, 15, 16, 17, 18, 19, 20, 21],
       [12, 13, 14, 15, 16, 17, 18, 19, 20, 21],
       [12, 13, 14, 15, 16, 17, 18, 19, 20, 21],
       [12, 13, 14, 15, 16, 17, 18, 19, 20, 21],
       [12, 13, 14, 15, 16, 17, 18, 19, 20, 21],
       [12, 13, 14, 15, 16, 17, 18, 19, 20, 21],
       [12, 13, 14, 15, 16, 17, 18, 19, 20, 21],
       [12, 13, 14, 15, 16, 17, 18, 19, 20, 21]])

In [4]:
dealer_count

array([[ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
       [ 2,  2,  2,  2,  2,  2,  2,  2,  2,  2],
       [ 3,  3,  3,  3,  3,  3,  3,  3,  3,  3],
       [ 4,  4,  4,  4,  4,  4,  4,  4,  4,  4],
       [ 5,  5,  5,  5,  5,  5,  5,  5,  5,  5],
       [ 6,  6,  6,  6,  6,  6,  6,  6,  6,  6],
       [ 7,  7,  7,  7,  7,  7,  7,  7,  7,  7],
       [ 8,  8,  8,  8,  8,  8,  8,  8,  8,  8],
       [ 9,  9,  9,  9,  9,  9,  9,  9,  9,  9],
       [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]])

In [ ]:
value_grid

In [ ]:
policy_grid

De code hieronder is om deze grids te plotten:

In [ ]:
def create_plots(value_grid, policy_grid, title: str):
    """Creates a plot using a value and policy grid."""
    # figuur met 2 subplots (links: state values, rechts: policy)
    player_count, dealer_count, value = value_grid
    fig = plt.figure(figsize=plt.figaspect(0.4))
    fig.suptitle(title, fontsize=16)

    # plot the state values
    ax1 = fig.add_subplot(1, 2, 1, projection="3d")
    ax1.plot_surface(
        player_count,
        dealer_count,
        value,
        rstride=1,
        cstride=1,
        cmap="viridis",
        edgecolor="none",
    )
    plt.xticks(range(12, 22), range(12, 22))
    plt.yticks(range(1, 11), ["A"] + list(range(2, 11)))
    ax1.set_title(f"State values: {title}")
    ax1.set_xlabel("Speler totaal")
    ax1.set_ylabel("Dealer toont")
    ax1.zaxis.set_rotate_label(False)
    ax1.set_zlabel("Waarde", fontsize=14, rotation=90)
    ax1.view_init(20, 220)

    # plot the policy
    fig.add_subplot(1, 2, 2)
    ax2 = sns.heatmap(policy_grid, linewidth=0, annot=True, cmap="Accent_r", cbar=False)
    ax2.set_title(f"Policy: {title}")
    ax2.set_xlabel("Speler totaal")
    ax2.set_ylabel("Waarde")
    ax2.set_xticklabels(range(12, 22))
    ax2.set_yticklabels(["A"] + list(range(2, 11)), fontsize=12) # A foor de Ace die gecodeerd is als 1

    # legende
    legend_elements = [
        Patch(facecolor="lightgreen", edgecolor="black", label="Hit"),
        Patch(facecolor="grey", edgecolor="black", label="Stick"),
    ]
    ax2.legend(handles=legend_elements, bbox_to_anchor=(1.3, 1))
    return fig


In [ ]:
# state values & policy with usable ace (ace counts as 11)
value_grid, policy_grid = create_grids(agent, usable_ace=True)
fig1 = create_plots(value_grid, policy_grid, title="With usable ace")
plt.show()

In [ ]:
# state values & policy without usable ace (ace counts as 1)
value_grid, policy_grid = create_grids(agent, usable_ace=False)
fig2 = create_plots(value_grid, policy_grid, title="Without usable ace")
plt.show()

Je kan best env.close() gebruiken aan het einde van je script, zodat alle resources terug vrijkomen.


In [ ]:
env.close()

## Kan je beter doen ?

In [5]:
# Visualiseer de omgeving met de play functie, probeer een paar spellen te winnen